# Auction Sales Report — Bronze to Silver

This notebook processes the raw auction summary data from the **bronze** layer (`company_db.bronze.auctionsummary`) and delivers cleaned, typed records into the **silver** layer (`company_db.silver.auction_sales_report`).

**What it does:**
1. Loads shared environment variables and configuration from the setup notebook.
2. Reads raw auction summary data from the bronze table.
3. Converts `start_date`, `end_date`, and `published_date` to proper `DATE` types.
4. Renames `id` to `auction_id` so the business key is explicit and consistent.
5. Creates the silver Delta table if it does not already exist.
6. Merges transformed records into silver using `auction_id` as the match key — updating existing rows and inserting new ones with audit timestamps.
7. Validates the final silver table output.

**Source:** `company_db.bronze.auctionsummary`  
**Target:** `company_db.silver.auction_sales_report` @ `abfss://auction@company.dfs.core.windows.net/silver/auction_sales_report`

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Read the raw auction summary dataset from the bronze layer.
# This is the source DataFrame that will be standardized and merged into the silver table.
auction_sales_report = spark.table(f'{catalog}.bronze.auctionsummary').filter(F.col("batch_id") == batch_id)

In [ ]:
from src.transformations.auction_summary import transform_auction_summary

In [ ]:
auction_sales_report = auction_sales_report.withColumn(
    "auction_category",
    F.when(F.col("auction").rlike("(?i)benefit"), F.lit("benefits"))
     .otherwise(F.expr("ai_classify(auction, ARRAY('collectibles', 'art', 'charity'))"))
)

In [ ]:
auction_sales_report  = transform_auction_summary(auction_sales_report)

In [ ]:
# Create the silver Delta table if it does not already exist.
# This table stores cleaned and typed auction summary records for downstream reporting.

spark.sql(
f"""CREATE TABLE IF NOT EXISTS {catalog}.silver.auction_sales_report
(
    platform STRING,
    auction_id INT,
    auction STRING,
    start_date DATE,
    end_date DATE,
    published_date DATE,
    type STRING,
    lots INT,
    lots_sold INT,
    lots_with_bid INT,
    lots_sold_online INT,
    currency STRING,
    hammer_price INT,
    sold_online DOUBLE,
    absentee_bids INT,
    absentee_bid_value DOUBLE,
    total_online_bids INT,
    registered_bidders INT,
    winning_bidders INT,
    under_bidders INT,
    online_bidders INT,
    approved_bidders INT,
    page_views INT,
    clicks STRING,
    path STRING,
    ingestion_timestamp TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    batch_id STRING,
    auction_category STRING
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/silver/auction_sales_report';""")

This notebook builds the silver table `company_db.silver.auction_sales_report` from the bronze source `company_db.bronze.auctionsummary`.

Silver-layer logic:
* Load shared environment variables and configuration from the setup notebook.
* Read raw auction summary data from bronze.
* Standardize key date columns to proper `DATE` values.
* Rename `id` to `auction_id` so the business key is clear and consistent.
* Create the silver Delta table if it does not already exist.
* Merge transformed records into silver using `auction_id` as the match key.
* Stamp audit fields with `created_at` and `updated_at` timestamps.
* Query the final silver table to validate the load.

Data dictionary:

| Column | Type | Description |
| --- | --- | --- |
| `platform` | string | Auction platform or source system. |
| `auction_id` | int | Unique auction identifier used as the merge key. |
| `auction` | string | Auction name or title. |
| `start_date` | date | Auction start date. |
| `end_date` | date | Auction end date. |
| `published_date` | date | Date the auction was published. |
| `type` | string | Auction type or category. |
| `lots` | int | Total number of lots listed. |
| `lots_sold` | int | Number of lots sold. |
| `lots_with_bid` | int | Number of lots that received bids. |
| `lots_sold_online` | int | Number of lots sold online. |
| `currency` | string | Currency for monetary values. |
| `hammer_price` | int | Final hammer price total. |
| `sold_online` | double | Online sold value or ratio from the source. |
| `absentee_bids` | int | Count of absentee bids. |
| `absentee_bid_value` | double | Total value of absentee bids. |
| `total_online_bids` | int | Total number of online bids. |
| `registered_bidders` | int | Number of registered bidders. |
| `winning_bidders` | int | Number of winning bidders. |
| `under_bidders` | int | Number of underbidders. |
| `online_bidders` | int | Number of online bidders. |
| `approved_bidders` | int | Number of approved bidders. |
| `page_views` | int | Auction page view count. |
| `clicks` | string | Click activity captured from the source. |
| `path` | string | Source or page path associated with the auction. |
| `ingestion_timestamp` | timestamp | Timestamp when the source row was ingested. |
| `created_at` | timestamp | Timestamp when the row was first inserted into silver. |
| `updated_at` | timestamp | Timestamp when the row was last updated in silver. |

In [ ]:
# Register the transformed DataFrame as a temporary view so the SQL MERGE statement
# can reference the cleaned dataset as its source.
auction_sales_report.createOrReplaceTempView("auction_sales_report_view")

In [ ]:
# Upsert the transformed auction records into the silver table.
# Existing rows are updated by auction_id, and new rows are inserted with audit timestamps.

spark.sql(
f"""
MERGE INTO {catalog}.silver.auction_sales_report AS silver
USING auction_sales_report_view AS new
ON silver.auction_id = new.auction_id
WHEN MATCHED 
AND silver.batch_id >= new.batch_id  
THEN
UPDATE SET
    silver.platform = new.platform,
    silver.auction_id = new.auction_id,
    silver.auction = new.auction,
    silver.start_date = new.start_date,
    silver.end_date = new.end_date,
    silver.published_date = new.published_date,
    silver.type = new.type,
    silver.lots = new.lots,
    silver.lots_sold = new.lots_sold,
    silver.lots_with_bid = new.lots_with_bid,
    silver.lots_sold_online = new.lots_sold_online,
    silver.currency = new.currency,
    silver.hammer_price = new.hammer_price,
    silver.sold_online = new.sold_online,
    silver.absentee_bids = new.absentee_bids,
    silver.absentee_bid_value = new.absentee_bid_value,
    silver.total_online_bids = new.total_online_bids,
    silver.registered_bidders = new.registered_bidders,
    silver.winning_bidders = new.winning_bidders,
    silver.under_bidders = new.under_bidders,
    silver.online_bidders = new.online_bidders,
    silver.approved_bidders = new.approved_bidders,
    silver.page_views = new.page_views,
    silver.clicks = new.clicks,
    silver.path = new.path,
    silver.ingestion_timestamp = new.ingestion_timestamp,
    silver.updated_at = current_timestamp(),
    silver.batch_id = new.batch_id,
    silver.auction_category = new.auction_category
WHEN NOT MATCHED THEN
    INSERT (
        platform, auction_id, auction, start_date, end_date, published_date, type, lots, lots_sold,
        lots_with_bid, lots_sold_online, currency, hammer_price, sold_online, absentee_bids,
        absentee_bid_value, total_online_bids, registered_bidders, winning_bidders, under_bidders,
        online_bidders, approved_bidders, page_views, clicks, path, ingestion_timestamp,
        created_at, updated_at,batch_id, auction_category
    )
    VALUES (
        new.platform, new.auction_id, new.auction, new.start_date, new.end_date, new.published_date,
        new.type, new.lots, new.lots_sold, new.lots_with_bid, new.lots_sold_online, new.currency,
        new.hammer_price, new.sold_online, new.absentee_bids, new.absentee_bid_value,
        new.total_online_bids, new.registered_bidders, new.winning_bidders, new.under_bidders,
        new.online_bidders, new.approved_bidders, new.page_views, new.clicks, new.path,
        new.ingestion_timestamp, current_timestamp(), current_timestamp(),new.batch_id,new.auction_category
    )""")